# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

Below, we enumerate the record sets and display all fields available in each, referencing every entity by its `@id`.

In [ ]:
# List all available record sets and their fields (by @id)
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = dataset.record_sets

record_set_ids = []

print("Available record sets:\n-----------------------")
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for f in getattr(rs, 'fields', []):
        print(f"    - Field name: {getattr(f, 'name', '<no name>')}")
        print(f"      @id: {f.id}")
    print("")
if not record_set_ids:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references are made using `@id`s.

Below, we extract each available record set into a pandas DataFrame and list their columns (fields by `@id`).

In [ ]:
# If no record sets discovered above, try to get from dataset interface
if not record_set_ids:
    # fallback: try to get record set ids from the loader
    record_sets_info = dataset.record_sets
    for rs in record_sets_info:
        record_set_ids.append(rs.id)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting data for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns for record set '{record_set_id}': {df.columns.tolist()}\n")
        print(df.head(3))
    else:
        print(f"No records found for record set {record_set_id}.\n")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and exploring relationships between attributes. All field and record set references are strictly by `@id`.

First, we pick a record set with at least one numeric field. We then apply some EDA steps:
- Filter rows by a threshold on a numeric field
- Normalize that field
- Group by a categorical field (if present)

*You may need to adjust `numeric_field_id` and `group_field_id` to match available @ids above.*

In [ ]:
# Choose a record set with data for EDA
if dataframes:
    # Select the first non-empty record set
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}\nColumns: {list(df.columns)}\n")

    # Try to select a numeric field
    # Simple heuristic: look for columns with int/float values or names typical for numeric fields
    numeric_field_id = None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
    if numeric_field_id is None:
        # try common names
        for c in df.columns:
            if any(s in c.lower() for s in ['age', 'count', 'interval', 'size', 'year', 'duration']):
                if pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors='coerce')):
                    numeric_field_id = c
                    break

    if numeric_field_id is not None:
        print(f"Selected numeric field: {numeric_field_id}")
        # Ensure the column is float
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized values for {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Look for a group field (categorical field)
        group_field_id = None
        for c in df.columns:
            if c != numeric_field_id and df[c].nunique() < 10 and df[c].dtype == object:
                group_field_id = c
                break
        if group_field_id is not None:
            print(f"\nGrouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped stats (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No DataFrames extracted from any record set.")

## 5. Visualization
Visualize the distribution of the selected numeric field (if available), and plots of relationship with a categorical field.

Below, we use matplotlib or seaborn for simple visualization of field values. All axes and legends are labeled by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group_field_id was found, display box plot
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable filtered DataFrame for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² dataset defined via a Croissant schema, leveraging record sets, fields, and columns referenced by their `@id`. We demonstrated how to extract tabular data, identify numeric and categorical fields using `mlcroissant`, and performed basic EDA and visualization steps.

**Next Steps**: To conduct domain-specific analyses, review the field descriptions via their `@id`s and adapt filtering, grouping, or modeling operations as needed.